In [1]:
# Import required libraries
import numpy as np
import scipy
import scipy.linalg as sla
import matplotlib.pyplot as plt

In [2]:
# Read the required file | Roll No: 23b2157
roll_no = "23b2157.npz"
with np.load(roll_no) as system:
    u = system["x"]
    f = system["b"]
    K = scipy.sparse.coo_matrix(
        (system["A_values"], list(system["A_indices"])),
        shape=(u.size, u.size)
    )

In [3]:
# Check K * u = f
if(np.allclose(K.dot(u), f)):
    print("Everything is correct!")
else:
    print("There is something wrong!")

print("K shape:",K.shape)
print("u shape:",u.shape)
print("f shape:",f.shape)

Everything is correct!
K shape: (1494, 1494)
u shape: (1494,)
f shape: (1494,)


## Dense Pipeline

In [4]:
K_dense = K.toarray()
n = K_dense.shape[0]

# Record the factorization time using timeit
svd_factor_time = %timeit -o sla.svd(K_dense)

1.12 s ± 122 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
# Dense SVD FLOPs - theoretical formula for full SVD varies, typically ~21*n^3 
flops_factor_svd = 21 * n**3
print("Factorization FlOps:", flops_factor_svd)

Factorization FlOps: 70027897464


In [6]:
U_mat, s, Vh = sla.svd(K_dense)
print("Average Time:", svd_factor_time)
print("Minimum Time (No noise):", svd_factor_time.best)

Average Time: 1.12 s ± 122 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Minimum Time (No noise): 1.014051125


In [7]:
# Solve the factorised matrices
# Kx = f => U * diag(s) * Vh * x = f => x = Vh.T * diag(1/s) * U.T * f
y = U_mat.T @ f
z = y / s
u_solved_svd = Vh.T @ z
residual_svd = np.linalg.norm(K_dense @ u_solved_svd - f)

In [8]:
print("Residual:", residual_svd)

Residual: 3.867165772543146e-08


In [9]:
# Record the time required to solve
svd_solve_time = %timeit -o Vh.T @ ((U_mat.T @ f) / s)

505 µs ± 2.99 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
print("Average Time:", svd_solve_time)
print("Minimum Time:", svd_solve_time.best)
# Solve FLOPs: 2n^2 (U^T * f) + n (division by s) + 2n^2 (Vh^T * z) = ~4n^2
print("FlOps:", 4 * n**2) 
print("GFlOpS:", (4 * n**2) / (svd_solve_time.average * 10**9))

Average Time: 505 µs ± 2.99 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Minimum Time: 0.0005028457910000003
FlOps: 8928144
GFlOpS: 17.666041355819033


In [11]:
# Memory storage for Dense matrices
mem_dense_bytes = K_dense.nbytes
mem_U_bytes = U_mat.nbytes
mem_s_bytes = s.nbytes
mem_Vh_bytes = Vh.nbytes

print("Dense matrix storage (MB):", mem_dense_bytes / 1024**2)
print("U storage (MB):", mem_U_bytes / 1024**2)
print("s storage (MB):", mem_s_bytes / 1024**2)
print("Vh storage (MB):", mem_Vh_bytes / 1024**2)

Dense matrix storage (MB): 17.029083251953125
U storage (MB): 17.029083251953125
s storage (MB): 0.0113983154296875
Vh storage (MB): 17.029083251953125
